In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.manual_seed(0)
device = "cuda:0"

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Targeting Device: {device.upper()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
hf_model.eval()

cfg = hf_model.config
N_LAYERS = cfg.num_hidden_layers
N_HEADS = cfg.num_attention_heads
N_KV_HEADS = getattr(cfg, "num_key_value_heads", N_HEADS)
HEAD_DIM = cfg.hidden_size // N_HEADS
KV_GROUPS = N_HEADS // N_KV_HEADS

ROPE_THETA = getattr(cfg, "rope_theta", None)
if ROPE_THETA is None and getattr(cfg, "rope_scaling", None):
    ROPE_THETA = cfg.rope_scaling.get("rope_theta")
assert ROPE_THETA is not None, "couldn't find rope_theta anywhere on cfg -- inspect cfg directly"

def rope_freqs(dim, base):
    assert dim % 2 == 0
    i = torch.arange(0, dim // 2, dtype=torch.float32)
    return base ** (-2.0 * i / dim)

FREQS = rope_freqs(HEAD_DIM, ROPE_THETA)

print(f"layers={N_LAYERS} heads={N_HEADS} kv_heads={N_KV_HEADS} head_dim={HEAD_DIM} "
      f"rope_theta={ROPE_THETA} model_device={next(hf_model.parameters()).device}")

expected = ROPE_THETA ** (-2.0 / HEAD_DIM)
actual = FREQS[1].item()
assert abs(expected - actual) < 1e-4, f"FREQS/ROPE_THETA mismatch: expected {expected}, got {actual}"
print(f"FREQS verified against ROPE_THETA={ROPE_THETA}: OK")

Targeting Device: CPU


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

layers=24 heads=14 kv_heads=2 head_dim=64 rope_theta=1000000.0 model_device=cpu
FREQS verified against ROPE_THETA=1000000.0: OK


In [ ]:
def rope_freqs(dim, base):
    assert dim % 2 == 0
    i = torch.arange(0, dim // 2, dtype=torch.float32)
    return base ** (-2.0 * i / dim)
FREQS = rope_freqs(HEAD_DIM, ROPE_THETA)
def apply_rope(x, positions, freqs):
    """x: [..., T, Dh]; positions: [T]."""
    current_device = x.device

    if not isinstance(positions, torch.Tensor):
        positions = torch.tensor(positions, device=current_device)
    else:
        positions = positions.to(current_device)

    freqs = freqs.to(current_device) if isinstance(freqs, torch.Tensor) else freqs

    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]

    angles = positions.unsqueeze(-1) * freqs
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    while cos.dim() < x1.dim():
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)

    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos
    return torch.cat([out1, out2], dim=-1)

def repeat_kv(x, n_rep):
    """[B, n_kv_heads, T, Dh] -> [B, n_kv_heads * n_rep, T, Dh]"""
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    x = x[:, :, None, :, :].expand(B, H, n_rep, T, D)
    return x.reshape(B, H * n_rep, T, D)

In [ ]:
@torch.no_grad()
def layer_step(layer, x, position, kv_cache):
    """x: [1,1,D] single new token's hidden state. kv_cache: dict with
    'k','v' shape [1, n_kv_heads, Tc, Dh] or None. Returns (new_x, new_kv_cache)."""
    attn = layer.self_attn
    residual = x
    h = layer.input_layernorm(x)
    q = attn.q_proj(h).view(1, 1, N_HEADS, HEAD_DIM).transpose(1, 2)
    k = attn.k_proj(h).view(1, 1, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    v = attn.v_proj(h).view(1, 1, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    pos_t = torch.tensor([position], dtype=torch.float32)
    q = apply_rope(q, pos_t, FREQS)
    k = apply_rope(k, pos_t, FREQS)
    if kv_cache is not None and kv_cache["k"].shape[2] > 0:
        k_full = torch.cat([kv_cache["k"], k], dim=2)
        v_full = torch.cat([kv_cache["v"], v], dim=2)
    else:
        k_full, v_full = k, v
    k_rep = repeat_kv(k_full, KV_GROUPS)
    v_rep = repeat_kv(v_full, KV_GROUPS)
    att = (q @ k_rep.transpose(-2, -1)) / (HEAD_DIM ** 0.5)
    att = F.softmax(att, dim=-1)
    out = att @ v_rep
    out = out.transpose(1, 2).reshape(1, 1, N_HEADS * HEAD_DIM)
    out = attn.o_proj(out)
    x = residual + out
    residual = x
    h = layer.post_attention_layernorm(x)
    x = residual + layer.mlp(h)
    return x, {"k": k_full, "v": v_full}
@torch.no_grad()
def model_step(token_id, position, kv_caches):
    """token_id: python int. kv_caches: list of per-layer dicts or Nones.
    Returns (logits[vocab], new_kv_caches)."""
    current_device = next(hf_model.parameters()).device

    token_tensor = torch.tensor([[token_id]], device=current_device)

    x = hf_model.model.embed_tokens(token_tensor)

    new_caches = []
    for layer, cache in zip(hf_model.model.layers, kv_caches):
        x, new_cache = layer_step(layer, x, position, cache)
        new_caches.append(new_cache)

    x = hf_model.model.norm(x)
    logits = hf_model.lm_head(x)[0, 0]
    return logits, new_caches

def empty_caches(n_layers):
    return [None] * n_layers
def concat_cache(a, b):
    if a is None or a["k"].shape[2] == 0:
        return b
    if b is None or b["k"].shape[2] == 0:
        return a
    return {"k": torch.cat([a["k"], b["k"]], dim=2), "v": torch.cat([a["v"], b["v"]], dim=2)}
def slice_cache(cache, start=0, end=None):
    if cache is None:
        return None
    return {"k": cache["k"][:, :, start:end, :], "v": cache["v"][:, :, start:end, :]}
@torch.no_grad()
def rerotate_cache(kv_caches, delta):
    """Exact corrective rotation applied to cached K only -- never V, never sink."""
    out = []
    for cache in kv_caches:
        if cache is None or cache["k"].shape[2] == 0:
            out.append(cache)
            continue
        Tc = cache["k"].shape[2]
        delta_t = torch.full((Tc,), float(delta))
        out.append({"k": apply_rope(cache["k"], delta_t, FREQS), "v": cache["v"]})
    return out

In [8]:
FILLER_WORDS = ["the", "cat", "sat", "on", "mat", "and", "ran", "far", "away",
                "into", "town", "market", "river", "bridge", "forest", "path"]

In [9]:
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

FILLER_WORD_IDS = {w: tokenizer(" " + w, add_special_tokens=False)["input_ids"][0] for w in FILLER_WORDS}
_QUERY_BASE_IDS = tokenizer("The secret number is", add_special_tokens=False)["input_ids"]
_SPACE_ID = tokenizer(" ", add_special_tokens=False)["input_ids"][0]

def make_multi_cycle_stream_fast(rng, n_cycles, cycle_len, sink_size=6):
    magic_number = rng.integers(2, 9)
    magic_ids = tokenizer(f"The secret number is {magic_number}.", add_special_tokens=False)["input_ids"]
    answer_id = tokenizer(str(magic_number), add_special_tokens=False)["input_ids"][0]

    words = list(FILLER_WORD_IDS.keys())
    ids = [FILLER_WORD_IDS[w] for w in words]

    sink_filler = [ids[i] for i in rng.integers(0, len(words), size=sink_size)]
    stream = [(tid, False, None) for tid in sink_filler]
    stream += [(tid, False, None) for tid in magic_ids]

    n_filler = n_cycles * cycle_len
    for i in rng.integers(0, len(words), size=n_filler):
        stream.append((ids[i], False, None))

    for tid in _QUERY_BASE_IDS:
        stream.append((tid, False, None))
    stream.append((_SPACE_ID, True, answer_id))
    return stream, len(magic_ids), magic_number


In [ ]:
@torch.inference_mode()
def _run_prefix_until_first_eviction(trial_stream, fact_len, sink_size, window_size, block_size):
    sink_caches = empty_caches(N_LAYERS)
    fact_caches = empty_caches(N_LAYERS)
    window_caches = empty_caches(N_LAYERS)
    sink_tokens, window_tokens = [], []
    logical_pos = 0
    n_fact_tokens_seen = 0

    for idx, (tid, is_query, answer_id) in enumerate(trial_stream):
        if is_query:
            return dict(done=True, sink_caches=sink_caches, fact_caches=fact_caches,
                        window_caches=window_caches, window_tokens=window_tokens,
                        logical_pos=logical_pos, resume_idx=idx)

        logical_pos += 1
        base = [concat_cache(s, concat_cache(f, w)) for s, f, w in zip(sink_caches, fact_caches, window_caches)]

        if len(sink_tokens) < sink_size:
            sink_tokens.append(tid)
            _, new_full = model_step(tid, logical_pos, base)
            sink_caches = [slice_cache(c, start=0, end=len(sink_tokens)) for c in new_full]
        elif n_fact_tokens_seen < fact_len:
            _, new_full = model_step(tid, logical_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact_prior = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            fact_caches = [slice_cache(c, start=n_sink, end=n_sink + n_fact_prior + 1) for c in new_full]
            n_fact_tokens_seen += 1
        else:
            _, new_full = model_step(tid, logical_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            window_tokens.append(tid)
            window_caches = [slice_cache(c, start=n_sink + n_fact) for c in new_full]

            if len(window_tokens) > 0 and len(window_tokens) % block_size == 0 and len(window_tokens) > window_size:
                return dict(done=False, sink_caches=sink_caches, fact_caches=fact_caches,
                            window_caches=window_caches, window_tokens=window_tokens,
                            logical_pos=logical_pos, resume_idx=idx + 1)

    return dict(done=True, sink_caches=sink_caches, fact_caches=fact_caches,
                window_caches=window_caches, window_tokens=window_tokens,
                logical_pos=logical_pos, resume_idx=idx)


@torch.inference_mode()
def _run_from_state(trial_stream, resume_idx, state, sink_size, window_size, block_size, apply_correction):
    sink_caches, fact_caches = state["sink_caches"], state["fact_caches"]
    window_caches = state["window_caches"]
    window_tokens = list(state["window_tokens"])
    logical_pos = state["logical_pos"]
    result = {"correct": None, "answer_logprob": None, "n_evictions": 0, "tokens_reprocessed": 0}

    for tid, is_query, answer_id in trial_stream[resume_idx:]:
        base = [concat_cache(s, concat_cache(f, w)) for s, f, w in zip(sink_caches, fact_caches, window_caches)]
        if is_query:
            logical_pos += 1
            logits, _ = model_step(tid, logical_pos, base)
            logical_pos -= 1
            result["correct"] = (logits.argmax(-1).item() == answer_id)
            result["answer_logprob"] = F.log_softmax(logits, dim=-1)[answer_id].item()
            continue

        logical_pos += 1
        _, new_full = model_step(tid, logical_pos, base)
        n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
        n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
        window_tokens.append(tid)
        window_caches = [slice_cache(c, start=n_sink + n_fact) for c in new_full]

        if len(window_tokens) > 0 and len(window_tokens) % block_size == 0 and len(window_tokens) > window_size:
            evict_n = block_size
            n_survivors = window_caches[0]["k"].shape[2] - evict_n if window_caches[0] is not None else 0
            window_tokens = window_tokens[evict_n:]
            window_caches = [slice_cache(c, start=evict_n) for c in window_caches]
            if apply_correction:
                window_caches = rerotate_cache(window_caches, -evict_n)
                result["tokens_reprocessed"] += max(n_survivors, 0)
                logical_pos -= evict_n
            result["n_evictions"] += 1

    return result


def run_strategy_b_pair_fast(trial_stream, fact_len, sink_size=6, window_size=24, block_size=8):
    """Returns (result_corrected, result_uncorrected), sharing all compute
    up to the first eviction instead of redoing it twice."""
    state = _run_prefix_until_first_eviction(trial_stream, fact_len, sink_size, window_size, block_size)

    if state["done"]:
        tid, is_query, answer_id = trial_stream[state["resume_idx"]]
        base = [concat_cache(s, concat_cache(f, w)) for s, f, w in
                zip(state["sink_caches"], state["fact_caches"], state["window_caches"])]
        logits, _ = model_step(tid, state["logical_pos"] + 1, base)
        r = {"correct": (logits.argmax(-1).item() == answer_id),
             "answer_logprob": F.log_softmax(logits, dim=-1)[answer_id].item(), "n_evictions": 0}
        return dict(r), dict(r)

    def clone_caches(caches):
        return [None if c is None else {"k": c["k"].clone(), "v": c["v"].clone()} for c in caches]

    state_c = dict(state, sink_caches=clone_caches(state["sink_caches"]),
                   fact_caches=clone_caches(state["fact_caches"]),
                   window_caches=clone_caches(state["window_caches"]), window_tokens=list(state["window_tokens"]))
    state_u = dict(state, sink_caches=clone_caches(state["sink_caches"]),
                   fact_caches=clone_caches(state["fact_caches"]),
                   window_caches=clone_caches(state["window_caches"]), window_tokens=list(state["window_tokens"]))

    r_c = _run_from_state(trial_stream, state["resume_idx"], state_c, sink_size, window_size, block_size, True)
    r_u = _run_from_state(trial_stream, state["resume_idx"], state_u, sink_size, window_size, block_size, False)
    return r_c, r_u



In [ ]:
@torch.no_grad()
def run_strategy_a_fact_persistence(trial_stream, fact_len, sink_size=6, window_size=24, horizon=48):
    sink_caches = empty_caches(N_LAYERS)
    fact_caches = empty_caches(N_LAYERS)
    window_caches = empty_caches(N_LAYERS)
    sink_tokens, fact_tokens, window_tokens = [], [], []
    true_pos = 0
    n_fact_tokens_seen = 0
    result = {"correct": None, "answer_logprob": None, "n_consolidations": 0, "tokens_reprocessed": 0}

    def replay_all(sink_ids, fact_ids, window_ids):
        caches = empty_caches(N_LAYERS)
        pos = 0
        for tid in sink_ids:
            pos += 1
            _, caches = model_step(tid, pos, caches)
        n_sink = len(sink_ids)
        sink_c = [slice_cache(c, start=0, end=n_sink) for c in caches]
        for tid in fact_ids:
            pos += 1
            _, caches = model_step(tid, pos, caches)
        n_fact = len(fact_ids)
        fact_c = [slice_cache(c, start=n_sink, end=n_sink + n_fact) for c in caches]
        for tid in window_ids:
            pos += 1
            _, caches = model_step(tid, pos, caches)
        window_c = [slice_cache(c, start=n_sink + n_fact) for c in caches]
        # Every single token above went through a real forward pass again.
        result["tokens_reprocessed"] += len(sink_ids) + len(fact_ids) + len(window_ids)
        return sink_c, fact_c, window_c, pos

    for tid, is_query, answer_id in trial_stream:
        if is_query:
            base = [concat_cache(s, concat_cache(f, w)) for s, f, w in zip(sink_caches, fact_caches, window_caches)]
            true_pos += 1
            logits, _ = model_step(tid, true_pos, base)
            true_pos -= 1
            result["correct"] = (logits.argmax(-1).item() == answer_id)
            result["answer_logprob"] = F.log_softmax(logits, dim=-1)[answer_id].item()
            continue

        true_pos += 1
        base = [concat_cache(s, concat_cache(f, w)) for s, f, w in zip(sink_caches, fact_caches, window_caches)]

        if len(sink_tokens) < sink_size:
            sink_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            sink_caches = [slice_cache(c, start=0, end=len(sink_tokens)) for c in new_full]

        elif n_fact_tokens_seen < fact_len:
            fact_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact_prior = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            fact_caches = [slice_cache(c, start=n_sink, end=n_sink + n_fact_prior + 1) for c in new_full]
            n_fact_tokens_seen += 1

        else:
            window_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            window_caches = [slice_cache(c, start=n_sink + n_fact) for c in new_full]

            if len(window_tokens) > window_size:
                window_tokens.pop(0)
                window_caches = [slice_cache(c, start=1) for c in window_caches]

            dist = true_pos - (sink_size + fact_len - 1)
            if dist >= horizon:
                sink_caches, fact_caches, window_caches, true_pos = replay_all(
                    sink_tokens, fact_tokens, window_tokens
                )
                result["n_consolidations"] += 1

    return result

In [12]:
def sanitize_row(row):
    sanitized = {}
    for k, v in row.items():
        if isinstance(v, bool):
            sanitized[k] = v
        elif isinstance(v, float):
            sanitized[k] = None if math.isnan(v) else v
        elif hasattr(v, "item"):
            sanitized[k] = v.item()
        elif isinstance(v, (np.integer, np.int64)):
            sanitized[k] = int(v)
        else:
            sanitized[k] = v
    return sanitized

In [16]:
NAMES = ["Alice", "Bob", "Carol", "Dave", "Eve"]

def make_multi_fact_stream_fast(rng, n_cycles, cycle_len, n_facts=3, sink_size=6):
    numbers = rng.choice(range(2, 9), size=n_facts, replace=False).tolist()
    names = rng.choice(NAMES, size=n_facts, replace=False).tolist()
    query_idx = rng.integers(0, n_facts)
    query_name = names[query_idx]
    correct_number = numbers[query_idx]

    fact_ids = []
    for name, num in zip(names, numbers):
        fact_ids.extend(tokenizer(f"{name}'s secret number is {num}.", add_special_tokens=False)["input_ids"])
    fact_len = len(fact_ids)

    words = list(FILLER_WORD_IDS.keys())
    ids = [FILLER_WORD_IDS[w] for w in words]
    sink_filler = [ids[i] for i in rng.integers(0, len(words), size=sink_size)]
    stream = [(tid, False, None) for tid in sink_filler]
    stream += [(tid, False, None) for tid in fact_ids]

    n_filler = n_cycles * cycle_len
    for i in rng.integers(0, len(words), size=n_filler):
        stream.append((ids[i], False, None))

    query_base = f"{query_name}'s secret number is"
    query_base_ids = tokenizer(query_base, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(query_base + " " + str(correct_number), add_special_tokens=False)["input_ids"]
    assert full_ids[len(query_base_ids)] == _SPACE_ID
    answer_id = full_ids[len(query_base_ids) + 1]

    for tid in query_base_ids:
        stream.append((tid, False, None))
    stream.append((_SPACE_ID, True, answer_id))
    return stream, fact_len, correct_number

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm

completed = set()
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            completed.add((row["n_cycles"], row["seed"], row["trial_index"]))
    print(f"Resuming: found {len(completed)} already-completed trials in {LOG_PATH}")
else:
    print("No existing log found -- starting fresh")

CYCLE_LEN = 16
n_facts = 3
N_TRIALS_PER_K = 50
SEEDS = [0, 1, 2]
cycle_values = [2, 4, 8, 12, 16]

summary = []

for n_cycles in tqdm(cycle_values, desc="Sweeping Cycle Values"):
    trial_results_for_summary = []
    for seed in SEEDS:
        rng = np.random.default_rng(seed * 1000 + n_cycles)
        for trial_index in range(N_TRIALS_PER_K):
            stream, fact_len, correct_number = make_multi_fact_stream_fast(
                rng, n_cycles, CYCLE_LEN, n_facts, sink_size=6
            )

            key = (n_cycles, seed, trial_index)
            if key in completed:
                continue 

            r_c, r_u = run_strategy_b_pair_fast(stream, fact_len=fact_len, sink_size=6, window_size=24, block_size=8)
            r_a = run_strategy_a_fact_persistence(stream, fact_len, sink_size=6, window_size=24, horizon=48)

            row = {
                "n_cycles": n_cycles, "seed": seed, "trial_index": trial_index,
                "correct_number": correct_number,
                "correct_B_corrected": r_c["correct"], "logprob_B_corrected": r_c["answer_logprob"],
                "n_evictions_corrected": r_c.get("n_evictions", 0), "tokens_reprocessed_B": r_c.get("tokens_reprocessed", 0),
                "correct_B_uncorrected": r_u["correct"], "logprob_B_uncorrected": r_u["answer_logprob"],
                "n_evictions_uncorrected": r_u.get("n_evictions", 0),
                "correct_A": r_a["correct"], "logprob_A": r_a["answer_logprob"],
                "n_consolidations_A": r_a["n_consolidations"], "tokens_reprocessed_A": r_a.get("tokens_reprocessed", 0),
            }
            sanitized = sanitize_row(row)
            with open(LOG_PATH, "a") as f:
                f.write(json.dumps(sanitized) + "\n")
                f.flush() 
            completed.add(key)  

    with open(LOG_PATH) as f:
        all_rows = [json.loads(l) for l in f if l.strip()]
    rows_here = [r for r in all_rows if r["n_cycles"] == n_cycles]

    acc_c = np.mean([r["correct_B_corrected"] for r in rows_here])
    acc_u = np.mean([r["correct_B_uncorrected"] for r in rows_here])
    acc_a = np.mean([r["correct_A"] for r in rows_here])
    mean_ev = np.mean([r["n_evictions_corrected"] for r in rows_here])
    mean_tok_a = np.mean([r["tokens_reprocessed_A"] for r in rows_here])
    mean_tok_b = np.mean([r["tokens_reprocessed_B"] for r in rows_here])
    summary.append((n_cycles, acc_a, acc_c, acc_u))
    print(f"n_cycles={n_cycles:3d} | A: {acc_a*100:5.1f}% | B-corr: {acc_c*100:5.1f}% | B-uncorr: {acc_u*100:5.1f}% "
          f"| n={len(rows_here)} | mean_evictions: {mean_ev:.1f} | tokens A/B: {mean_tok_a:.0f}/{mean_tok_b:.0f}")
    if mean_ev < 1:
        print(f"  ⚠️  WARNING: mean evictions < 1 -- this row tested almost nothing")

print(f"\nTotal trials logged: {len(completed)}")


Resuming: found 411 already-completed trials in /content/drive/MyDrive/colab_experiments/per_trial_log_3way.jsonl


Sweeping Cycle Values:  40%|████      | 2/5 [00:00<00:00, 12.90it/s]

n_cycles=  2 | A:  88.7% | B-corr:  92.0% | B-uncorr:  92.0% | n=150 | mean_evictions: 0.0 | tokens A/B: 0/0
  ⚠️  WARNING: mean evictions < 1 -- this row tested almost nothing
n_cycles=  4 | A:  84.0% | B-corr:  82.7% | B-uncorr:  90.0% | n=150 | mean_evictions: 4.0 | tokens A/B: 66/128


Sweeping Cycle Values:  60%|██████    | 3/5 [15:14<12:42, 381.12s/it]

n_cycles=  8 | A:  90.0% | B-corr:  89.3% | B-uncorr:  95.3% | n=150 | mean_evictions: 12.0 | tokens A/B: 218/384


Sweeping Cycle Values:  80%|████████  | 4/5 [1:43:15<36:49, 2209.26s/it]

n_cycles= 12 | A:  87.3% | B-corr:  88.7% | B-uncorr:  96.0% | n=150 | mean_evictions: 20.0 | tokens A/B: 382/640


Sweeping Cycle Values: 100%|██████████| 5/5 [3:43:41<00:00, 2684.28s/it]

n_cycles= 16 | A:  90.7% | B-corr:  91.3% | B-uncorr:  95.3% | n=150 | mean_evictions: 28.0 | tokens A/B: 546/896

Total trials logged: 750


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/colab_experiments"
os.makedirs(DRIVE_DIR, exist_ok=True)

LOG_PATH = os.path.join(DRIVE_DIR, "per_trial_log_3way.jsonl")
print(f"Log file successfully mapped to Google Drive: {LOG_PATH}")


Mounted at /content/drive
Log file successfully mapped to Google Drive: /content/drive/MyDrive/colab_experiments/per_trial_log_3way.jsonl


In [13]:
test_content = torch.randn(1, N_KV_HEADS, 1, HEAD_DIM, device=device)
P, evict_n = 40, 8

rotated_then_corrected = apply_rope(test_content, torch.tensor([float(P)], device=device), FREQS)
rotated_then_corrected = apply_rope(rotated_then_corrected, torch.tensor([float(-evict_n)], device=device), FREQS)

fresh_at_target_position = apply_rope(test_content, torch.tensor([float(P - evict_n)], device=device), FREQS)

print("max abs diff:", (rotated_then_corrected - fresh_at_target_position).abs().max().item())

max abs diff: 4.76837158203125e-07
